In [2]:
import pandas as pd
import numpy as np 
from matplotlib import pyplot as plt
from sklearn import preprocessing 
from sklearn.preprocessing import StandardScaler , MinMaxScaler , RobustScaler ,PowerTransformer
from sklearn .model_selection  import train_test_split
from sklearn .linear_model import LinearRegression


In [3]:
df = pd.read_csv("Housing - Housing.csv")

In [4]:
df .info

<bound method DataFrame.info of         price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0    13300000  7420         4          2        3      yes        no       no   
1    12250000  8960         4          4        4      yes        no       no   
2    12250000  9960         3          2        2      yes        no      yes   
3    12215000  7500         4          2        2      yes        no      yes   
4    11410000  7420         4          1        2      yes       yes      yes   
..        ...   ...       ...        ...      ...      ...       ...      ...   
540   1820000  3000         2          1        1      yes        no      yes   
541   1767150  2400         3          1        1       no        no       no   
542   1750000  3620         2          1        1      yes        no       no   
543   1750000  2910         3          1        1       no        no       no   
544   1750000  3850         3          1        2      yes        no       no

In [7]:
df.head(10)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000,5750,3,2,4,yes,yes,no,no,yes,1,yes,unfurnished


In [ ]:
def capping_outliers(df,columns: list,method:str='iqr',z:float=3,whisker_length:float=1.5,inplace:bool=False)->tuple:
    """
    Capping outliers in the specified columns of a DataFrame using either the IQR method or Z-score method.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns (list): List of column names to cap outliers.
    method (str): Method to use for capping ('iqr' or 'zscore'). Default is 'iqr'.
    z (float): Z-score threshold for capping when using the 'zscore' method. Default is 3.

    Returns:
    pd.DataFrame: DataFrame with capped outliers.
    """
    report = []
    df_capped = df.copy()
    
    if method == 'iqr':
        for col in columns:
            Q1 = df_capped[col].quantile(0.25)
            Q3 = df_capped[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - whisker_length * IQR
            upper_bound = Q3 + whisker_length * IQR
            
            df_capped[col] = np.where(df_capped[col] < lower_bound, lower_bound, 
                                       np.where(df_capped[col] > upper_bound, upper_bound, df_capped[col]))
    
    elif method == 'zscore':
        for col in columns:
            mean = df_capped[col].mean()
            std_dev = df_capped[col].std()
            lower_bound = mean - z * std_dev
            upper_bound = mean + z * std_dev
            
            df_capped[col] = np.where(df_capped[col] < lower_bound, lower_bound, 
                                       np.where(df_capped[col] > upper_bound, upper_bound, df_capped[col]))

            
    
    else:
        raise ValueError("Method must be either 'iqr' or 'zscore'.")

    n_outliers=(df_capped[columns] < lower_bound).sum().sum() + (df_capped[columns] > upper_bound).sum().sum()

    report.append({
        'columns': columns,
        'method': method,
        'n_outliers': n_outliers,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    })

    return df_capped, pd.DataFrame(report)